In [ ]:
import pandas as pd
columns_to_read = ["FROM_YEAR", "AGE", "SEX","ICD_DIAG_01_PRIMARY","ICD_DIAG_02","ICD_DIAG_03","ICD_DIAG_04",
                   "ICD_DIAG_05","ICD_DIAG_06","ICD_DIAG_07","ICD_DIAG_08","ICD_DIAG_09","ICD_DIAG_10","ICD_DIAG_11",
                   "ICD_DIAG_12","ICD_DIAG_13"]

files = [
    "PUBLICUSE_CLAIM_MC_2020.txt",
    "PUBLICUSE_CLAIM_MC_2021.txt",
    "PUBLICUSE_CLAIM_MC_2022.txt",
    "PUBLICUSE_CLAIM_MC_2023.txt",
    "PUBLICUSE_CLAIM_MC_2024.txt"
]

chunk_size = 10_000
combined_chunks = []

# Read each file in chunks
for file in files:
    for chunk in pd.read_csv(file, sep="|", chunksize=chunk_size, usecols=columns_to_read):
        combined_chunks.append(chunk)

# Concatenate all chunks into a single DataFrame
df = pd.concat(combined_chunks, ignore_index=True)

print(df.info())

In [ ]:
df.columns

In [ ]:
df

In [ ]:
df.describe(include='all')


In [ ]:
df['SEX'].describe()

In [ ]:
count_m = (df['SEX'] == 'M').sum()
count_m

In [ ]:
other_count = ~df['SEX'].isin(['M', 'F'])
print(other_count.sum())

In [ ]:
#change to int and string to nan
df['AGE'] = df['AGE'].replace('90+', 90)
df['AGE'] = df['AGE'].astype(int)
df['AGE'].describe()

In [ ]:
df[df['FROM_YEAR']==2024]

# longitudinal data loading and processing

In [ ]:
import pandas as pd

# Load the uploaded file
file_path = 'icd_code_pairs_longitudinal_2020_2024_sanitized.csv'
data = pd.read_csv(file_path)

# Display the first few rows to understand the structure of the data
data.head()


In [ ]:
# Identify the unique ICD codes in the dataset to isolate respiratory virus-related diseases
# Split the pairs into individual codes and create a unique list
data['ICD_Code_1'] = data['ICD_Code_Pair'].apply(lambda x: eval(x)[0])
data['ICD_Code_2'] = data['ICD_Code_Pair'].apply(lambda x: eval(x)[1])

unique_icd_codes = pd.unique(data[['ICD_Code_1', 'ICD_Code_2']].values.ravel())

# Display the unique ICD codes to identify respiratory virus-related diseases
unique_icd_codes[:30]  # Display the first 30 unique codes for review


In [ ]:
# Filter the dataset for respiratory virus-related diseases (e.g., codes starting with 'J')
# Assuming respiratory-related codes follow the ICD-10 convention starting with 'J'
respiratory_related_codes = [code for code in unique_icd_codes if code.startswith('J')]

# Filter the dataset for pairs involving respiratory codes
respiratory_data = data[
    data['ICD_Code_1'].isin(respiratory_related_codes) |
    data['ICD_Code_2'].isin(respiratory_related_codes)
]

# Display the filtered dataset to confirm
respiratory_data.head()


In [ ]:
respiratory_data.head(20)

In [ ]:
respiratory_data.head(20).to_csv('respiratory_data.csv', index=False)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Data: ICD code pairs and their counts
data = [
    ('E785', 'J449'), ('E785', 'J45909'), ('F419', 'J45909'), ('E119', 'J449'),
    ('E119', 'J45909'), ('E039', 'J45909'), ('E1122', 'J449'), ('F329', 'J45909'),
    ('E039', 'J449'), ('F419', 'J449'), ('F17210', 'J449'), ('E6601', 'J45909'),
    ('E669', 'J45909'), ('F329', 'J449'), ('E876', 'J45909'), ('D649', 'J449'),
    ('E7800', 'J45909'), ('E1151', 'J449'), ('E871', 'J449'), ('E876', 'J449')
]

# Sample ICD code to description mapping (replace with actual descriptions if available)
icd_descriptions = {
    'E785': 'Hyperlipidemia',
    'J449': 'Chronic obstructive \npulmonary disease, unspecified',
    'J45909': 'Unspecified asthma, uncomplicated',
    'F419': 'Anxiety disorder, unspecified',
    'E119': 'Type 2 diabetes without complications',
    'E039': 'Hypothyroidism, unspecified',
    'E1122': 'Type 2 diabetes with kidney complications',
    'F329': 'Major depressive disorder, single episode, unspecified',
    'F17210': 'Nicotine dependence, cigarettes, uncomplicated',
    'E6601': 'Obesity due to \nexcess calories',
    'E669': 'Obesity, unspecified',
    'D649': 'Anemia, unspecified',
    'E7800': 'Pure hypercholesterolemia',
    'E1151': 'Type 2 diabetes with diabetic \n peripheral angiopathy',
    'E871': 'Hypopotassemia',
    'E876': 'Hyperpotassemia'
}

# Create a graph
G = nx.Graph()

# Add edges to the graph
for pair in data:
    G.add_edge(pair[0], pair[1])

# Create labels with ICD codes and descriptions
labels = {code: f"{code}\n{icd_descriptions.get(code, 'Description not available')}" for code in G.nodes()}

# Use Kamada-Kawai layout for better spacing
pos = nx.kamada_kawai_layout(G)

# Draw the graph
plt.figure(figsize=(14, 12))

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=3000, alpha=0.9)

# Draw edges with curvature
nx.draw_networkx_edges(G, pos, edge_color='gray', width=1.5, alpha=0.7, connectionstyle='arc3,rad=0.1')

# Draw labels with adjusted font size and bbox
nx.draw_networkx_labels(G, pos, labels, font_size=10, font_weight='bold', 
                        bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5'))

# Remove axes for a cleaner look
plt.axis('off')

# Add a title
#plt.title("Highly Correlated Disease Pairs (ICD Codes with Descriptions)", fontsize=16, pad=20)

# Show the graph
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# Assuming the data is already loaded into a DataFrame called respiratory_data
# respiratory_data = pd.read_csv('your_data.csv')  # Load your data here

# Function to calculate the correlation between two time series
def calculate_correlation(series1, series2):
    return pearsonr(series1, series2)[0]

# Function to analyze temporal changes and identify potential causal relationships
def analyze_causal_relationships(data):
    results = []
    
    # Iterate over each ICD code pair
    for index, row in data.iterrows():
        icd_pair = row['ICD_Code_Pair']
        counts_2020 = row['Count_2020']
        counts_2021 = row['Count_2021']
        counts_2022 = row['Count_2022']
        counts_2023 = row['Count_2023']
        counts_2024 = row['Count_2024']
        
        # Create a time series for each ICD code in the pair
        time_series_1 = [counts_2020, counts_2021, counts_2022, counts_2023, counts_2024]
        time_series_2 = [counts_2020, counts_2021, counts_2022, counts_2023, counts_2024]
        
        # Calculate the correlation between the two time series
        correlation = calculate_correlation(time_series_1, time_series_2)
        
        # Store the results
        results.append({
            'ICD_Code_Pair': icd_pair,
            'Correlation': correlation,
            'ICD_Code_1': row['ICD_Code_1'],
            'ICD_Code_2': row['ICD_Code_2']
        })
    
    # Convert results to a DataFrame
    results_df = pd.DataFrame(results)
    
    # Sort by absolute correlation value to identify the strongest relationships
    results_df['Abs_Correlation'] = results_df['Correlation'].abs()
    results_df = results_df.sort_values(by='Abs_Correlation', ascending=False)
    
    return results_df

# Perform the analysis
causal_relationships = analyze_causal_relationships(respiratory_data)

# Display the top 10 potential causal relationships
print(causal_relationships)

In [ ]:
from pgmpy.estimators import PC
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import BicScore
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# Prepare data for causal discovery
# Extract co-occurrence counts for each pair across the years and transpose
causal_data = respiratory_data[
    ['Count_2020', 'Count_2021', 'Count_2022', 'Count_2023', 'Count_2024']
]

# Ensure data is in a pandas DataFrame
causal_data = pd.DataFrame(causal_data, columns=[
    'Count_2020', 'Count_2021', 'Count_2022', 'Count_2023', 'Count_2024'
])

# Apply the PC algorithm for causal discovery
# Use BIC (Bayesian Information Criterion) as the score function
pc_estimator = PC(data=causal_data)
model = pc_estimator.estimate(significance_level=0.05)  # Adjust significance level as needed

# Visualize the resulting causal graph
# The model itself is a DAG that can be used directly with NetworkX
graph = nx.DiGraph(model.edges())

# Plot the graph
plt.figure(figsize=(10, 8))
nx.draw(graph, with_labels=True, node_size=3000, node_color="lightblue", font_size=12)
plt.title("Causal Graph of Respiratory Diseases")
plt.show()

# Save the causal graph as an image
causal_graph_path = "causal_graph_respiratory_diseases.png"
plt.savefig(causal_graph_path)

causal_graph_path


## DAG graph

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Define the directed acyclic graph (DAG) structure
dag = nx.DiGraph()

# Define node categories and their colors
node_categories = {
    'risk_factors': ['Hypertension', 'Hyperlipidemia', 'GERD', 'Diabetes', 'Systemic Inflammation'],
    'chronic_conditions': ['Chronic Kidney Disease', 'Asthma', 'COPD'],
    'acute_conditions': ['Pneumonia', 'Acute Respiratory Failure']
}

node_colors = {
    'risk_factors': '#ADD8E6',      # Light blue
    'chronic_conditions': '#98FB98', # Light green
    'acute_conditions': '#FFB6C1'    # Light pink
}

# Flatten nodes list while preserving category order
nodes = []
for category in node_categories.values():
    nodes.extend(category)

# Define edges
edges = [
    ("Hypertension", "COPD"),
    ("Hyperlipidemia", "COPD"),
    ("Hyperlipidemia", "Acute Respiratory Failure"),
    ("GERD", "Asthma"),
    ("Diabetes", "Chronic Kidney Disease"),
    ("Chronic Kidney Disease", "Acute Respiratory Failure"),
    ("COPD", "Pneumonia"),
    ("Pneumonia", "Acute Respiratory Failure"),
    ("Systemic Inflammation", "COPD"),
    ("Systemic Inflammation", "Asthma"),
    ("Systemic Inflammation", "Acute Respiratory Failure")
]

# Add nodes and edges
dag.add_nodes_from(nodes)
dag.add_edges_from(edges)

# Create figure with white background
plt.figure(figsize=(15, 10), facecolor='white')  # Changed to white
ax = plt.gca()
ax.set_facecolor('white')  # Changed to white

# Use hierarchical layout instead of spring layout
pos = nx.kamada_kawai_layout(dag)

# Adjust y-coordinates to create more distinct levels
for node in pos:
    if node in node_categories['risk_factors']:
        pos[node][1] += 0.3  # Move risk factors up
    elif node in node_categories['acute_conditions']:
        pos[node][1] -= 0.3  # Move acute conditions down

# Draw edges with enhanced arrows
for edge in edges:
    nx.draw_networkx_edges(dag, pos,
                          edgelist=[edge],
                          edge_color='#4682B4',
                          width=3,
                          alpha=0.7,
                          arrowsize=25,
                          arrowstyle='-|>',
                          connectionstyle='arc3,rad=0.2',
                          min_target_margin=30,
                          min_source_margin=20)

# Draw nodes with category-specific colors and effects
for category, category_nodes in node_categories.items():
    nx.draw_networkx_nodes(dag, pos,
                          nodelist=category_nodes,
                          node_size=4000,
                          node_color=node_colors[category],
                          edgecolors='#4682B4',
                          linewidths=2,
                          alpha=0.9)

# Add labels with improved styling
labels = {node: node for node in nodes}
nx.draw_networkx_labels(dag, pos,
                       labels,
                       font_size=11,
                       font_weight='bold',
                       font_family='sans-serif',
                       bbox=dict(facecolor='white',
                               edgecolor='#4682B4',
                               boxstyle='round,pad=0.5',
                               alpha=0.8))

# Add title and legend
plt.title("Figure 1. Causal Pathways in Multimorbidity",
         fontsize=18,
         fontweight='bold',
         fontfamily='sans-serif',
         pad=20)

# Add legend with improved positioning
legend_elements = [plt.Line2D([0], [0], marker='o', color='w', 
                             markerfacecolor=color, markersize=15,
                             label=category.replace('_', ' ').title())
                  for category, color in node_colors.items()]

legend_elements.append(plt.Line2D([0], [0], color='#4682B4', 
                                 marker='>', markersize=15,
                                 label='Causal Direction'))

plt.legend(handles=legend_elements,
          loc='upper left',
          bbox_to_anchor=(1, 1),
          fontsize=12,
          title='Legend',
          title_fontsize=14)

# Final styling
plt.margins(0.2)
plt.tight_layout()
plt.axis('off')
plt.show()

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

# Create DataFrame from the data
data = pd.DataFrame([
    ['E785-I10', 170447, 181586, 17595, 143666, 54365],
    ['E785-Z79899', 165776, 163316, 15897, 189473, 70620],
    ['E785-Z87891', 145589, 148628, 12470, 125475, 46286],
    ['E119-I10', 125735, 119386, 11421, 116090, 42607],
    ['E785-Z7982', 123493, 108016, 8877, 97337, 34117],
    ['E785-K219', 117171, 119840, 9228, 106601, 43127],
    ['D509-D631', 115542, 118058, 7667, 101676, 18379],
    ['E119-E785', 107198, 96972, 8757, 94424, 33493],
    ['F419-F329', 88253, 71550, 1269, 8911, 2685]
], columns=['ICD_Pair', '2020', '2021', '2022', '2023', '2024'])

# Create a mapping of ICD codes to conditions
icd_mapping = {
    'E785': 'Hyperlipidemia',
    'I10': 'Hypertension',
    'Z79899': 'Other Drug Therapy',
    'Z87891': 'History of Nicotine Dependence',
    'E119': 'Type 2 Diabetes',
    'Z7982': 'Long-term Drug Therapy',
    'K219': 'GERD',
    'D509': 'Iron Deficiency Anemia',
    'D631': 'Anemia in Chronic Disease',
    'F419': 'Anxiety',
    'F329': 'Depression'
}

# Create directed graph
G = nx.DiGraph()

# Add edges based on temporal patterns and frequency
for _, row in data.iterrows():
    codes = row['ICD_Pair'].split('-')
    if len(codes) == 2:
        code1, code2 = codes
        # Calculate average co-occurrence
        avg_occurrence = np.mean([row['2020'], row['2021'], row['2022'], row['2023'], row['2024']])
        
        # Add edge with weight based on average occurrence
        if code1 in icd_mapping and code2 in icd_mapping:
            G.add_edge(icd_mapping[code1], icd_mapping[code2], weight=avg_occurrence)

# Draw the network
plt.figure(figsize=(15, 10), facecolor='white')
pos = nx.spring_layout(G, k=2, iterations=50)

# Draw edges with width proportional to weight
edge_weights = [G[u][v]['weight']/5000 for u, v in G.edges()]
nx.draw_networkx_edges(G, pos, width=edge_weights, edge_color='#4682B4', 
                      arrowsize=20, arrowstyle='-|>')

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_size=3000, node_color='lightblue',
                      edgecolors='#4682B4', linewidths=2)

# Add labels
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold',
                       bbox=dict(facecolor='white', edgecolor='#4682B4',
                               boxstyle='round,pad=0.5', alpha=0.8))

plt.title("Disease Relationship Network\nBased on 2020-2024 Co-occurrence Patterns",
          fontsize=16, pad=20)
plt.axis('off')
plt.tight_layout()
plt.show()

# Calculate temporal correlations
def calculate_temporal_patterns(data):
    patterns = []
    for _, row in data.iterrows():
        codes = row['ICD_Pair'].split('-')
        if len(codes) == 2:
            code1, code2 = codes
            if code1 in icd_mapping and code2 in icd_mapping:
                condition1 = icd_mapping[code1]
                condition2 = icd_mapping[code2]
                years_data = row[['2020', '2021', '2022', '2023', '2024']].values
                
                # Calculate year-over-year changes
                changes = np.diff(years_data)
                consistency = np.mean(np.sign(changes) == np.sign(changes[0]))
                
                patterns.append({
                    'Primary': condition1,
                    'Secondary': condition2,
                    'Avg_Occurrence': np.mean(years_data),
                    'Pattern_Consistency': consistency
                })
    
    return pd.DataFrame(patterns)

temporal_patterns = calculate_temporal_patterns(data)
print("\nTemporal Pattern Analysis:")
print(temporal_patterns.sort_values('Avg_Occurrence', ascending=False))

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Define the directed acyclic graph (DAG) structure
dag = nx.DiGraph()

# Nodes
nodes = [
    "Hypertension", 
    "Hyperlipidemia", 
    "GERD", 
    "Diabetes", 
    "Chronic Kidney Disease",
    "Asthma", 
    "COPD", 
    "Pneumonia", 
    "Acute Respiratory Failure", 
    "Systemic Inflammation"
]

# Edges representing causal pathways
edges = [
    ("Hypertension", "COPD"),
    ("Hyperlipidemia", "COPD"),
    ("Hyperlipidemia", "Acute Respiratory Failure"),
    ("GERD", "Asthma"),
    ("Diabetes", "Chronic Kidney Disease"),
    ("Chronic Kidney Disease", "Acute Respiratory Failure"),
    ("COPD", "Pneumonia"),
    ("Pneumonia", "Acute Respiratory Failure"),
    ("Systemic Inflammation", "COPD"),
    ("Systemic Inflammation", "Asthma"),
    ("Systemic Inflammation", "Acute Respiratory Failure")
]

# Add nodes and edges to the graph
dag.add_nodes_from(nodes)
dag.add_edges_from(edges)

plt.figure(figsize=(12, 8))
pos = nx.spring_layout(dag, seed=42, k=1)  # Increased k for more spacing

# Draw nodes
nx.draw_networkx_nodes(dag, pos, 
                      node_size=3000, 
                      node_color="lightblue", 
                      edgecolors="black",
                      linewidths=2)

# Draw edges with enhanced arrows
nx.draw_networkx_edges(dag, pos,
                      arrowsize=80,  # Increased arrow size
                      arrowstyle='->',  # Changed arrow style
                      edge_color="black",
                      width=2,
                      connectionstyle='arc3,rad=0.2')  # Add curve to edges

# Draw labels with white background for better readability
labels = {node: node for node in nodes}
nx.draw_networkx_labels(dag, pos, 
                       labels,
                       font_size=10,
                       font_weight='bold',
                       bbox=dict(facecolor='white', 
                                edgecolor='none', 
                                alpha=0.7,
                                pad=2))

plt.title("Figure 1. Causal Pathways in Multimorbidity", 
         fontsize=16,
         pad=20)
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from itertools import combinations
from collections import Counter

# List of ICD diagnosis columns
icd_diag_columns = [
    'ICD_DIAG_02', 'ICD_DIAG_03', 'ICD_DIAG_04', 'ICD_DIAG_05', 
    'ICD_DIAG_06', 'ICD_DIAG_07', 'ICD_DIAG_08', 'ICD_DIAG_09', 
    'ICD_DIAG_10', 'ICD_DIAG_11', 'ICD_DIAG_12', 'ICD_DIAG_13'
]

# Function to extract all unique ICD code pairs in a single row
def extract_pairs(row):
    codes = row.dropna().unique()  # Drop NaN values and get unique codes
    return list(combinations(codes, 2))  # Generate all possible pairs

# Function to count ICD code pairs for a specific year
def count_icd_pairs_by_year(data, year):
    icd_data_year = data[data['FROM_YEAR'] == year][icd_diag_columns]
    all_pairs = icd_data_year.apply(extract_pairs, axis=1).explode().dropna()
    pair_counts = Counter(all_pairs)
    return pd.DataFrame(pair_counts.items(), columns=["ICD_Code_Pair", f"Count_{year}"])

# Process data for all years (2020–2024)
years = range(2020, 2025)
yearly_counts = []

for year in years:
    yearly_counts.append(count_icd_pairs_by_year(df, year))

# Merge all yearly data into a single DataFrame
merged_pairs = yearly_counts[0]

for i in range(1, len(yearly_counts)):
    merged_pairs = pd.merge(
        merged_pairs, yearly_counts[i], on="ICD_Code_Pair", how="outer"
    ).fillna(0)

# Calculate temporal changes
for year in years[1:]:
    prev_year = f"Count_{year - 1}"
    curr_year = f"Count_{year}"
    merged_pairs[f"Change_{prev_year}_to_{curr_year}"] = (
        merged_pairs[curr_year] - merged_pairs[prev_year]
    )

# Save the longitudinal analysis to a CSV
merged_pairs.to_csv("icd_code_pairs_longitudinal_2020_2024.csv", index=False)

# Display some results
print("Top temporal changes in ICD code pairs:")
print(merged_pairs.sort_values(by=f"Change_{years[-2]}_to_{years[-1]}", ascending=False).head(10))


In [ ]:
import pandas as pd
from itertools import combinations
from collections import Counter

# List of ICD diagnosis columns
icd_diag_columns = [
    'ICD_DIAG_02', 'ICD_DIAG_03', 'ICD_DIAG_04', 'ICD_DIAG_05', 
    'ICD_DIAG_06', 'ICD_DIAG_07', 'ICD_DIAG_08', 'ICD_DIAG_09', 
    'ICD_DIAG_10', 'ICD_DIAG_11', 'ICD_DIAG_12', 'ICD_DIAG_13'
]

# Function to extract all unique ICD code pairs in a single row
def extract_pairs(row):
    codes = row.dropna().unique()  # Drop NaN values and get unique codes
    return list(combinations(codes, 2))  # Generate all possible pairs

# Function to count ICD code pairs for a specific year
def count_icd_pairs_by_year(data, year):
    icd_data_year = data[data['FROM_YEAR'] == year][icd_diag_columns]
    all_pairs = icd_data_year.apply(extract_pairs, axis=1).explode().dropna()
    pair_counts = Counter(all_pairs)
    return pd.DataFrame(pair_counts.items(), columns=["ICD_Code_Pair", f"Count_{year}"])

# Count pairs for each year
pairs_2023 = count_icd_pairs_by_year(df, 2023)
pairs_2024 = count_icd_pairs_by_year(df, 2024)

# Merge the two datasets for comparison
merged_pairs = pd.merge(
    pairs_2023, pairs_2024, on="ICD_Code_Pair", how="outer"
).fillna(0)

# Add a column for the difference in counts between the years
merged_pairs['Difference'] = merged_pairs['Count_2024'] - merged_pairs['Count_2023']

# Save the results to a CSV
merged_pairs.to_csv("icd_code_pairs_longitudinal.csv", index=False)

# Display top changes
print("Top changes in ICD code pairs:")
print(merged_pairs.sort_values(by="Difference", ascending=False).head(10))


In [ ]:
# cover 2020 - 2024 
import pandas as pd
from itertools import combinations
from collections import Counter

# List of ICD diagnosis columns
icd_diag_columns = [
    'ICD_DIAG_02', 'ICD_DIAG_03', 'ICD_DIAG_04', 'ICD_DIAG_05', 
    'ICD_DIAG_06', 'ICD_DIAG_07', 'ICD_DIAG_08', 'ICD_DIAG_09', 
    'ICD_DIAG_10', 'ICD_DIAG_11', 'ICD_DIAG_12', 'ICD_DIAG_13'
]

# Function to extract all unique ICD code pairs in a single row
def extract_pairs(row):
    codes = row.dropna().unique()  # Drop NaN values and get unique codes
    return list(combinations(codes, 2))  # Generate all possible pairs

# Function to count ICD code pairs for a specific year
def count_icd_pairs_by_year(data, year):
    icd_data_year = data[data['FROM_YEAR'] == year][icd_diag_columns]
    all_pairs = icd_data_year.apply(extract_pairs, axis=1).explode().dropna()
    pair_counts = Counter(all_pairs)
    return pd.DataFrame(pair_counts.items(), columns=["ICD_Code_Pair", f"Count_{year}"])

# Process data for all years (2020–2024)
years = range(2020, 2025)
yearly_counts = []

for year in years:
    yearly_counts.append(count_icd_pairs_by_year(df, year))

# Merge all yearly data into a single DataFrame
merged_pairs = yearly_counts[0]

for i in range(1, len(yearly_counts)):
    merged_pairs = pd.merge(
        merged_pairs, yearly_counts[i], on="ICD_Code_Pair", how="outer"
    ).fillna(0)

# Calculate temporal changes
for year in years[1:]:
    prev_year = f"Count_{year - 1}"
    curr_year = f"Count_{year}"
    merged_pairs[f"Change_{prev_year}_to_{curr_year}"] = (
        merged_pairs[curr_year] - merged_pairs[prev_year]
    )

# Save the longitudinal analysis to a CSV
merged_pairs.to_csv("icd_code_pairs_longitudinal_2020_2024.csv", index=False)

# Display some results
print("Top temporal changes in ICD code pairs:")
print(merged_pairs.sort_values(by=f"Change_{years[-2]}_to_{years[-1]}", ascending=False).head(10))


In [ ]:
import pandas as pd
from itertools import combinations
from collections import Counter

# List of ICD diagnosis columns
icd_diag_columns = [
    'ICD_DIAG_02', 'ICD_DIAG_03', 'ICD_DIAG_04', 'ICD_DIAG_05', 
    'ICD_DIAG_06', 'ICD_DIAG_07', 'ICD_DIAG_08', 'ICD_DIAG_09', 
    'ICD_DIAG_10', 'ICD_DIAG_11', 'ICD_DIAG_12', 'ICD_DIAG_13'
]

# Filter only the ICD diagnosis columns from the DataFrame
icd_data = df[icd_diag_columns]

# Function to extract all unique ICD code pairs in a single row
def extract_pairs(row):
    codes = row.dropna().unique()  # Drop NaN values and get unique codes
    return list(combinations(codes, 2))  # Generate all possible pairs

# Flatten all pairs across rows
all_pairs = icd_data.apply(extract_pairs, axis=1).explode().dropna()

# Count the frequency of each pair
pair_counts = Counter(all_pairs)

# Get the top 10 most common pairs
top_10_pairs = pair_counts.most_common(1000)

# Convert to DataFrame for easier viewing
top_10_pairs_df = pd.DataFrame(top_10_pairs, columns=["ICD_Code_Pair", "Count"])

# Display the results
print("Top 10 ICD code pairs:")
print(top_10_pairs_df)

# Save the results to a CSV
top_10_pairs_df.to_csv("top_10_icd_code_pairs.csv", index=False)


In [ ]:
# Get the top 10 most common pairs
top_10_pairs = pair_counts.most_common(1000)

# Convert to DataFrame for easier viewing
top_10_pairs_df = pd.DataFrame(top_10_pairs, columns=["ICD_Code_Pair", "Count"])

# Display the results
print("Top 10 ICD code pairs:")
print(top_10_pairs_df)

# Save the results to a CSV
top_10_pairs_df.to_csv("top_10_icd_code_pairs.csv", index=False)


In [ ]:
pair_counts